# W&B Sweep Report

Queries wandb sweep results and builds a summary table + Report. Self-contained: does not depend on `criteo_functions.py` or the other notebooks.

In [ ]:
import wandb
import polars as pl
import pandas as pd
import wandb_workspaces.reports.v2 as wr

In [ ]:
def get_project_sweep_ids(
    entity: str = "seanv507",
) -> dict[str, list[dict]]:
    """
    Return every accessible project under `entity` and its sweep IDs.

    Projects without sweeps are included with an empty list.
    """
    api = wandb.Api(timeout=60)

    return {
        project.name: [
            {
                "sweep_id": sweep.id,
                "sweep_config": sweep.config,
                "sweep_runs": [{
                    "run_id": run.id,
                    "run_summary_metrics": run.summary_metrics
                } for run in sweep.runs]
            }
            for sweep in project.sweeps(per_page=100)
        ]
        for project in api.projects(entity=entity, per_page=200)
    }

In [ ]:
wandb.login()

In [ ]:
api = wandb.Api(timeout=60)
sw = api.sweep('seanv507/criteo_boost_sweep_04/ozi2tmij')
sw.config

In [ ]:
run = api.run('seanv507/criteo_boost_sweep_04/auwure38')

In [ ]:
run.summary_metrics
["_runtime", 'validation_0-logloss.min', 'validation_1-logloss.min']

In [ ]:
run.summary_metrics

In [ ]:
ps = get_project_sweep_ids("seanv507")

In [ ]:
project_sweeps = (
    pl.DataFrame({'project':ps.keys(), 'sweeps': ps.values()},strict=False)
    .explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')
    .with_columns(run_id=pl.col('run_id'), training_logloss=pl.col('run_summary_metrics').struct['validation_0-logloss.min'], 
                     validation_logloss=pl.col('run_summary_metrics').struct['validation_1-logloss.min'])
    .drop_nulls(subset='validation_logloss')
    .sort('validation_logloss')
)


In [ ]:
project_sweeps.with_columns(pl.col('sweep_config').struct['parameters']).unnest('parameters')

In [ ]:
a1 = project_sweeps.explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')


> **⚠️ Syntax error below** — `a2 = a1.` is an incomplete statement (trailing `.` with nothing after it). It looks like it was meant to continue the explode/unnest chain from `a1` above. Complete the expression or remove the cell.

In [ ]:
a2 = a1.
a2.head()

In [ ]:
a2.shape

In [ ]:
a1.shape

In [ ]:
a1.item(0,3)

In [ ]:
project_sweeps_e = project_sweeps.explode("sweep_ids")
project_sweeps_e.head()

In [ ]:
project_sweeps_e.group_by(pl.col("project").str.slice(0,10)).len().sort("len",descending=True)

In [ ]:
ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")


In [ ]:
values.head()

In [ ]:


stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)


In [ ]:
stats

In [ ]:

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)



> **⚠️ Duplicate** — this cell redoes the entire pipeline from the `ENTITY = ...` cell above through the Report-building cell, self-contained in one cell. Same missing-`pandas`-import issue noted above applies here too.

In [ ]:

ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")

stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)
